In [ ]:
def main(datasources, start_date, end_date):

    import numpy as np
    import pandas as pd
    import xgboost as xgb
    import dai
    import warnings
    warnings.filterwarnings("ignore")


    TRAIN_START = "2019-01-01"
    TRAIN_END = "2023-12-31"


    bar_table = datasources["bar1m"]
    fin_table = datasources["financial"]


    ############################################################
    # 1. 日频量价因子
    ############################################################

    def build_price_features(sd, ed):

        sql = f"""
        SELECT
            date_trunc('day',date)::DATE as date,
            instrument,

            ARG_MIN(open,date) open,
            MAX(high) high,
            MIN(low) low,
            ARG_MAX(close,date) close,

            SUM(volume) volume,
            SUM(amount) amount

        FROM {bar_table}

        GROUP BY date,instrument
        """

        df = dai.query(
            sql,
            filters={
                "date":[
                    pd.to_datetime(sd)-pd.Timedelta(days=40),
                    pd.to_datetime(ed)
                ]
            }
        ).df()


        df=df.sort_values(
            ["instrument","date"]
        )


        g=df.groupby("instrument")


        #收益
        df["ret1"]=g.close.pct_change()

        df["ret5"]=(
            g.close.shift(0)/
            g.close.shift(5)-1
        )


        df["ret20"]=(
            g.close/
            g.close.shift(20)-1
        )


        #反转
        df["reverse5"]=-df["ret5"]


        #波动

        df["vol20"]=(
            g.ret1
            .rolling(20)
            .std()
            .reset_index(level=0,drop=True)
        )


        df["vol_ratio"]=(
            g.ret1
            .rolling(5)
            .std()
            .reset_index(level=0,drop=True)
            /
            (df["vol20"]+1e-8)
        )


        #成交量异常

        df["volume_ratio"] = (
            df.volume /
            (
            g.volume
            .rolling(20)
            .mean()
            .reset_index(level=0,drop=True)
            +1e-8
            )
        )


        #价格位置

        high20=(
            g.high
            .rolling(20)
            .max()
            .reset_index(level=0,drop=True)
        )

        low20=(
            g.low
            .rolling(20)
            .min()
            .reset_index(level=0,drop=True)
        )


        df["price_position"]=(
            df.close-low20
        )/(high20-low20+1e-8)


        #换手代理

        df["liquidity"] = (
            df.amount/
            (df.volume+1)
        )


        return df



    ############################################################
    # 2. 高频盘口因子
    ############################################################

    def build_hf_features(sd,ed):

        sql=f"""

        SELECT

        date_trunc('day',date)::DATE date,
        instrument,

        AVG(
        (bid_volume1-ask_volume1)/
        (bid_volume1+ask_volume1+1)
        ) obi,


        SUM(volume) hf_volume


        FROM {bar_table}

        GROUP BY date,instrument

        """

        try:

            hf=dai.query(
                sql,
                filters={
                    "date":[
                    pd.to_datetime(sd)-pd.Timedelta(days=5),
                    pd.to_datetime(ed)
                    ]
                }
            ).df()

        except:

            hf=pd.DataFrame(
                columns=[
                    "date",
                    "instrument",
                    "obi",
                    "hf_volume"
                ]
            )


        return hf



    ############################################################
    # 3. 财务质量因子
    ############################################################

    def build_fin(sd,ed):

        sql=f"""

        SELECT

        date,
        instrument,

        net_profit_to_parent_shareholders,
        total_equity_to_parent_shareholders,
        operating_revenue


        FROM {fin_table}

        WHERE category='lf'
        AND shift=0

        """

        fin=dai.query(
            sql,
            filters={
            "date":[
            pd.to_datetime(sd)-pd.Timedelta(days=365),
            pd.to_datetime(ed)
            ]
            }
        ).df()


        fin=fin.sort_values(
            ["instrument","date"]
        )


        fin=(
            fin
            .groupby("instrument")
            .apply(lambda x:x.ffill())
            .reset_index(drop=True)
        )


        fin["roe"]=(
            fin.net_profit_to_parent_shareholders/
            (fin.total_equity_to_parent_shareholders+1)
        )


        fin["growth"]=(
            fin.operating_revenue/
            fin.groupby("instrument")
            .operating_revenue.shift(4)
            -1
        )


        return fin



    ############################################################
    # 4. 合并特征
    ############################################################


    def dataset(sd,ed):


        price=build_price_features(sd,ed)

        hf=build_hf_features(sd,ed)

        fin=build_fin(sd,ed)


        df=price.merge(
            hf,
            on=[
            "date",
            "instrument"
            ],
            how="left"
        )


        df=df.merge(
            fin,
            on=[
            "date",
            "instrument"
            ],
            how="left"
        )


        df=df.sort_values(
            ["instrument","date"]
        )


        #未来5日收益标签

        df["label"]=(
            df.groupby("instrument")
            .close.shift(-5)
            /
            df.close
            -1
        )


        features=[

        "ret5",
        "ret20",
        "reverse5",
        "vol20",
        "vol_ratio",
        "volume_ratio",
        "price_position",
        "liquidity",
        "obi",
        "hf_volume",
        "roe",
        "growth"

        ]


        for c in features:

            df[c]=pd.to_numeric(
                df[c],
                errors="coerce"
            )

            df[c]=df[c].replace(
                [np.inf,-np.inf],
                np.nan
            )


        return df,features



    ############################################################
    # 5. 训练
    ############################################################


    train,features=dataset(
        TRAIN_START,
        TRAIN_END
    )


    train=train.dropna(
        subset=["label"]
    )


    train[features]=(
        train[features]
        .fillna(0)
    )


    model=xgb.XGBRegressor(

        n_estimators=300,

        max_depth=4,

        learning_rate=0.03,

        subsample=0.8,

        colsample_bytree=0.8,

        reg_lambda=10,

        objective="reg:squarederror",

        tree_method="hist",

        n_jobs=-1,

        random_state=1

    )


    model.fit(
        train[features],
        train.label
    )


    ############################################################
    # 6. 测试预测
    ############################################################


    test,features=dataset(
        start_date,
        end_date
    )


    test[features]=(
        test[features]
        .fillna(0)
    )


    test["factor"]=model.predict(
        test[features]
    )



    ############################################################
    # 7. 截面标准化
    ############################################################


    test["factor"]=(
        test
        .groupby("date")
        ["factor"]
        .transform(
            lambda x:
            (x-x.mean())/
            (x.std()+1e-8)
        )
    )


    ############################################################
    # 8. 成分股过滤
    ############################################################


    pool=dai.query(
        """
        SELECT date,instrument
        FROM bigalpha_2026_instruments
        """,
        filters={
            "date":[start_date,end_date]
        }
    ).df()


    result=test.merge(
        pool,
        on=[
        "date",
        "instrument"
        ],
        how="inner"
    )


    result=result[
        [
        "date",
        "instrument",
        "factor"
        ]
    ]


    result=result.dropna()


    return result



if __name__=="__main__":

    datasources={

        "bar1m":
        "bigalpha_2026_stock_bar1m",

        "financial":
        "bigalpha_2026_financial"

    }


    factor=main(
        datasources,
        "2024-01-01",
        "2024-12-31"
    )


    print(factor.head())